In [ ]:
import zipfile
import os

zip_file_path = '/content/SendAnywhere_240107.zip'
extraction_path = 'extracted_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Open the zip file and extract its contents
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' extracted to '{extraction_path}' successfully.")
print(f"Contents of '{extraction_path}':")
for item in os.listdir(extraction_path):
    print(f"- {item}")

'/content/SendAnywhere_240107.zip' extracted to 'extracted_data' successfully.
Contents of 'extracted_data':
- Timetable.xlsx
- Rulebook.pdf
- Events.csv
- PlacementGuide.pdf
- Faculty.xlsx


# 📚 LangChain RAG Pipeline — Load → Chunk → Embed → FAISS → Search → Persist

This notebook builds a complete Retrieval-Augmented-Generation (RAG) ingestion pipeline using the
**latest LangChain (v0.3+) modular package structure**:

- `langchain-core`
- `langchain-community` (document loaders + FAISS wrapper)
- `langchain-text-splitters` (chunking)
- `langchain-huggingface` (free, local embeddings — no API key required)
- `faiss-cpu` (vector index)

**Pipeline stages**

1. Install dependencies
2. Configure inputs (folder of mixed files: PDF / CSV / XLSX / TXT / DOCX)
3. Load documents with the right loader per file type
4. Chunk documents (`RecursiveCharacterTextSplitter`)
5. Generate embeddings (`HuggingFaceEmbeddings`, swappable for `OpenAIEmbeddings`)
6. Build a FAISS vector store
7. Run similarity search queries
8. Persist the FAISS index to disk (`save_local` **and** a raw `.pkl` dump) for later reuse
9. Reload the persisted index and confirm it works

> 💡 Swap in `OpenAIEmbeddings` / `ChatOpenAI` anywhere by uncommenting the marked cells if you'd
> rather use OpenAI instead of local HuggingFace models.


## 1. Install dependencies

In [ ]:
# Core LangChain packages (latest modular split — NOT the old monolithic `langchain` imports)
!pip install -q -U langchain langchain-core langchain-community langchain-text-splitters
!pip install -q -U langchain-huggingface sentence-transformers
!pip install -q -U faiss-cpu

# Loaders for the file types we'll ingest
!pip install -q -U pypdf                 # PDF
!pip install -q -U unstructured          # generic/robust text extraction
!pip install -q -U openpyxl              # XLSX
!pip install -q -U pandas                # CSV / XLSX handling

# Optional: uncomment if you want to use OpenAI embeddings/LLM instead of local HF models
# !pip install -q -U langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━

## 2. Imports

In [ ]:
import os
import glob
import pickle
from pathlib import Path

import pandas as pd

from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    UnstructuredExcelLoader,
    TextLoader,
    Docx2txtLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Optional OpenAI alternative:
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI

print("✅ Imports ready")


/tmp/ipykernel_670/4121961062.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


✅ Imports ready


## 3. Configure inputs

Point `SOURCE_DIR` at the folder containing your files. Based on the uploaded screenshot, that's
Google Colab's `extracted_data` folder (unzipped from `SendAnywhere_240107.zip`) containing:

- `Events.csv`
- `Faculty.xlsx`
- `PlacementGuide.pdf`
- `Rulebook.pdf`
- `Timetable.xlsx`

Adjust the path if your files live elsewhere.


In [ ]:
# --- CONFIG ---------------------------------------------------------------
SOURCE_DIR = "/content/extracted_data"     # <- change if needed
FAISS_INDEX_DIR = "/content/faiss_index"   # save_local() target directory
FAISS_PKL_PATH = "/content/faiss_store.pkl"  # raw pickle dump for later reuse

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # free, local, 384-dim
# ----------------------------------------------------------------------------

os.makedirs(FAISS_INDEX_DIR, exist_ok=True)
print("Source dir:", SOURCE_DIR)
print("Files found:")
for f in sorted(glob.glob(os.path.join(SOURCE_DIR, "*"))):
    print(" -", f)


Source dir: /content/extracted_data
Files found:
 - /content/extracted_data/Events.csv
 - /content/extracted_data/Faculty.xlsx
 - /content/extracted_data/PlacementGuide.pdf
 - /content/extracted_data/Rulebook.pdf
 - /content/extracted_data/Timetable.xlsx


## 4. Load documents (multi-format loader router)

Each file type needs its own LangChain loader. This function inspects the extension and routes
to the correct loader, tagging every resulting `Document` with a `source` and `file_type` in its
metadata (useful later for filtering search results).


In [ ]:
def load_file(file_path: str):
    """Load a single file into a list of LangChain Document objects, based on extension."""
    ext = Path(file_path).suffix.lower()
    docs = []

    try:
        if ext == ".pdf":
            docs = PyPDFLoader(file_path).load()

        elif ext == ".csv":
            docs = CSVLoader(file_path, encoding="utf-8").load()

        elif ext in (".xlsx", ".xls"):
            # UnstructuredExcelLoader gives readable text per sheet;
            # mode="elements" preserves row/column structure better for tabular data.
            docs = UnstructuredExcelLoader(file_path, mode="elements").load()

        elif ext == ".txt":
            docs = TextLoader(file_path, encoding="utf-8").load()

        elif ext == ".docx":
            docs = Docx2txtLoader(file_path).load()

        else:
            print(f"⚠️  Skipping unsupported file type: {file_path}")
            return []

        # enrich metadata
        for d in docs:
            d.metadata["source"] = os.path.basename(file_path)
            d.metadata["file_type"] = ext.lstrip(".")

        print(f"✅ Loaded {len(docs):3d} doc(s) from {os.path.basename(file_path)}")
        return docs

    except Exception as e:
        print(f"❌ Failed to load {file_path}: {e}")
        return []


all_documents = []
for file_path in sorted(glob.glob(os.path.join(SOURCE_DIR, "*"))):
    all_documents.extend(load_file(file_path))

print(f"\n📄 Total raw documents loaded: {len(all_documents)}")


✅ Loaded   5 doc(s) from Events.csv
❌ Failed to load /content/extracted_data/Faculty.xlsx: No module named 'msoffcrypto'
✅ Loaded   2 doc(s) from PlacementGuide.pdf
✅ Loaded   1 doc(s) from Rulebook.pdf
❌ Failed to load /content/extracted_data/Timetable.xlsx: No module named 'msoffcrypto'

📄 Total raw documents loaded: 8


## 5. Chunk documents

We use `RecursiveCharacterTextSplitter`, which tries to split on paragraph → sentence → word
boundaries (in that order) so chunks stay semantically coherent. `chunk_overlap` preserves context
across chunk boundaries, which improves retrieval quality.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(all_documents)

# add a stable chunk id for traceability
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"✂️  Split {len(all_documents)} documents into {len(chunks)} chunks")
print("\nSample chunk:\n" + "-" * 60)
if chunks:
    print(chunks[0].page_content[:500])
    print("\nMetadata:", chunks[0].metadata)


✂️  Split 8 documents into 8 chunks

Sample chunk:
------------------------------------------------------------
Event: AI Workshop
Date: 12-Aug-2026
Time: 10:00 AM
Venue: Seminar Hall
Coordinator: Dr. Ravi

Metadata: {'source': 'Events.csv', 'row': 0, 'file_type': 'csv', 'chunk_id': 0}


## 6. Create the embedding model

Using a free, local `sentence-transformers` model via `langchain_huggingface` — no API key needed
and it runs fine on Colab CPU. Swap to `OpenAIEmbeddings(model="text-embedding-3-small")` if you'd
rather use OpenAI (requires `OPENAI_API_KEY`).


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# --- OpenAI alternative -----------------------------------------------------
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# -----------------------------------------------------------------------------

# quick sanity check
test_vec = embeddings.embed_query("sanity check")
print(f"✅ Embedding model ready — vector dimension: {len(test_vec)}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready — vector dimension: 384


## 7. Build the FAISS vector store

`FAISS.from_documents` embeds every chunk and builds the index in one call.


In [ ]:
vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

print(f"✅ FAISS index built with {vector_store.index.ntotal} vectors")


✅ FAISS index built with 8 vectors


## 8. Run search queries

`similarity_search` returns the top-k most relevant chunks.
`similarity_search_with_score` also returns the L2 distance (lower = more similar).


In [ ]:
def run_query(query: str, k: int = 4):
    print(f"\n🔎 Query: {query!r}\n" + "=" * 70)
    results = vector_store.similarity_search_with_score(query, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\n[{rank}] score={score:.4f}  source={doc.metadata.get('source')}")
        print(doc.page_content[:300].replace("\n", " ") + "...")
    return results


# Example queries — replace with whatever's relevant to your documents
_ = run_query("What is the placement process for students?")
_ = run_query("What are the rules students must follow?")



🔎 Query: 'What is the placement process for students?'

[1] score=0.9879  source=PlacementGuide.pdf
Campus Placement Handbook 2026 Eligibility – Minimum CGPA: 7.0 – No active backlogs. – Minimum attendance: 75% Placement Process – Registration – Resume Verification – Aptitude Test – Technical Interview – HR Interview – Offer Letter Required Documents – Resume – Aadhaar Card – PAN Card – Semester M...

[2] score=0.9923  source=Rulebook.pdf
Student Academic Rulebook 2026 Attendance Rules – Minimum attendance required: 75% – Students below 75% attendance are not eligible to appear for semester examinations un- less approved by the Principal. – Attendance is calculated separately for every subject. Examination Rules – Mid Semester Exam: ...

[3] score=1.2986  source=Events.csv
Event: Placement Drive Date: 30-Aug-2026 Time: 9:00 AM Venue: Placement Cell Coordinator: Dr. Vinay...

[4] score=1.2998  source=PlacementGuide.pdf
Interview Tips – Be punctual. – Maintain eye contact. – Prepare DSA

## 9. Persist the FAISS index for later use

Two ways to save, both included:

1. **`save_local()`** — LangChain's built-in method. Writes `index.faiss` (the raw FAISS index)
   and `index.pkl` (the docstore + id mapping) into a folder. This is the **recommended** way to
   reload with `FAISS.load_local()`.
2. **Raw `pickle.dump()`** — a single `.pkl` file containing the entire `FAISS` object, in case you
   specifically need one portable pickle file (e.g. to move between environments without the FAISS
   C++ index file).


In [ ]:
# 1) LangChain-native save (creates index.faiss + index.pkl inside FAISS_INDEX_DIR)
vector_store.save_local(FAISS_INDEX_DIR)
print(f"✅ Saved FAISS index (save_local) to: {FAISS_INDEX_DIR}")
print("   Contents:", os.listdir(FAISS_INDEX_DIR))

# 2) Raw pickle of the whole vector store object
with open(FAISS_PKL_PATH, "wb") as f:
    pickle.dump(vector_store, f)
print(f"✅ Saved raw pickle of the FAISS vector store to: {FAISS_PKL_PATH}")
print(f"   File size: {os.path.getsize(FAISS_PKL_PATH) / 1024:.1f} KB")


✅ Saved FAISS index (save_local) to: /content/faiss_index
   Contents: ['index.pkl', 'index.faiss']
✅ Saved raw pickle of the FAISS vector store to: /content/faiss_store.pkl
   File size: 89646.5 KB


## 10. Reload the persisted index (verification)

This simulates a fresh session: load the index back from disk and confirm search still works —
without re-embedding anything.


In [ ]:
# --- Option A: reload via LangChain's save_local/load_local ---------------
reloaded_vector_store = FAISS.load_local(
    FAISS_INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,  # required since LangChain 0.1+ (we trust our own file)
)
print(f"✅ Reloaded FAISS index via load_local — {reloaded_vector_store.index.ntotal} vectors")

# --- Option B: reload via the raw pickle file ------------------------------
with open(FAISS_PKL_PATH, "rb") as f:
    reloaded_from_pkl = pickle.load(f)
print(f"✅ Reloaded FAISS index via pickle — {reloaded_from_pkl.index.ntotal} vectors")

# quick confirmation query on the reloaded store
results = reloaded_vector_store.similarity_search("timetable schedule", k=2)
for doc in results:
    print("\n-", doc.metadata.get("source"), "->", doc.page_content[:150].replace("\n", " "))


✅ Reloaded FAISS index via load_local — 8 vectors
✅ Reloaded FAISS index via pickle — 8 vectors

- Events.csv -> Event: Placement Drive Date: 30-Aug-2026 Time: 9:00 AM Venue: Placement Cell Coordinator: Dr. Vinay

- Events.csv -> Event: Sports Meet Date: 24-Aug-2026 Time: 8:30 AM Venue: Sports Ground Coordinator: Prof. Sneha


## 11. (Optional) Wrap as a retriever for a RAG chain

Once the FAISS store is loaded, turn it into a retriever and plug it into any LLM chain
(`ChatOpenAI`, `ChatAnthropic`, a local Ollama model, etc.).


In [ ]:
retriever = reloaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Example: plug into an LLM chain (uncomment and configure your LLM of choice)
# from langchain_openai import ChatOpenAI
# from langchain.chains import RetrievalQA
#
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)
# response = qa_chain.invoke({"query": "Summarize the rulebook in 3 bullet points"})
# print(response["result"])

print("✅ Retriever ready:", retriever)


✅ Retriever ready: tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78834f09ccb0> search_kwargs={'k': 4}


---
### ✅ Summary

| Stage | Artifact |
|---|---|
| Loaders | `PyPDFLoader`, `CSVLoader`, `UnstructuredExcelLoader`, `TextLoader`, `Docx2txtLoader` |
| Chunking | `RecursiveCharacterTextSplitter` (`chunk_size=1000`, `overlap=150`) |
| Embeddings | `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`, local, free) |
| Vector store | `FAISS` (`langchain_community.vectorstores.FAISS`) |
| Persistence | `save_local()` → `faiss_index/` **and** `faiss_store.pkl` |
| Reload | `FAISS.load_local()` / `pickle.load()` |

You now have a reusable FAISS index on disk — reload it anytime with the Section 10 cell without
re-running the embedding step.
